In [312]:
import warnings
import logging

# Suppress Hugging Face and other warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("datasets").setLevel(logging.ERROR)
logging.getLogger("evaluate").setLevel(logging.ERROR)



In [ ]:
# from huggingface_hub import login

# login("hf_LIdVfCZHKiakgvLRpHe")

hf_token = "hf_KCS=xANOAvce"

In [90]:
#!pip install langchain-google-genai langchain faiss-cpu sentence-transformers groq dspy-ai


In [140]:
import os
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.prompts import PromptTemplate
import requests

In [ ]:
# 1. Set keys
os.environ["GOOGLE_API_KEY"] = "grA"
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_KCSAvce"
os.environ["GROQ_API_KEY"] = "3Juv"

In [144]:
import pandas as pd

# 2. Load data (replace these with your files)
df_passages = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
df_test = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

In [202]:
df_test.head(10)

,question,answer,relevant_passage_ids
id,,,
0,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","[20598273, 6650562, 15829955, 15617541, 230011..."
1,List signaling molecules (ligands) that intera...,The 7 known EGFR ligands are: epidermal growt...,"[23821377, 24323361, 23382875, 22247333, 23787..."
2,Is the protein Papilin secreted?,"Yes, papilin is a secreted protein","[21784067, 19297413, 15094122, 7515725, 332004..."
3,Are long non coding RNAs spliced?,Long non coding RNAs appear to be spliced thro...,"[22955974, 21622663, 22707570, 22955988, 24285..."
4,Is RANKL secreted from the cells?,Receptor activator of nuclear factor κB ligand...,"[22867712, 23827649, 21618594, 23835909, 24265..."
5,Does metformin interfere thyroxine absorption?,No. There are not reported data indicating tha...,[26191653]
6,Which miRNAs could be used as potential biomar...,"miR-200a, miR-100, miR-141, miR-200b, miR-200c...","[23918241, 23621186, 22246341, 23978303, 23888..."
7,Which acetylcholinesterase inhibitors are used...,Pyridostigmine and neostygmine are acetylcholi...,"[21328290, 21133188, 15610702, 20663605, 21815..."
8,Has Denosumab (Prolia) been approved by FDA?,"Yes, Denosumab was approved by the FDA in 2010.","[24114694, 22540167, 21129866, 21170699, 23956..."


In [204]:
df_test.shape

(4719, 3)

In [145]:
df_passages = df_passages.reset_index()
df_passages = df_passages.reset_index()

df_passages = df_passages.rename(columns={'id': 'id'})
df_passages = df_passages.rename(columns={'id': 'id'})


In [146]:
df_passages.head(10)

,index,id,passage
0,0,9797,New data on viruses isolated from patients wit...
1,1,11906,We describe an improved method for detecting d...
2,2,16083,We have studied the effects of curare on respo...
3,3,23188,Kinetic and electrophoretic properties of 230-...
4,4,23469,Male Wistar specific-pathogen-free rats aged 2...
5,5,24032,Tyrosine hydroxylase (TH) and phenylethanolami...
6,6,30666,Hemolytic anemia is a well-recognized complica...
7,7,58611,(1) The RNA replicase induced by bacteriophage...
8,8,61441,Mice were inoculated with human sarcoid tissue...
9,9,83311,Bleomycin is potentially capable of inducing a...


In [150]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')
passages = df_passages['passage'].tolist()
ids = df_passages['id'].tolist()
passage_embeds = embedder.encode(passages, show_progress_bar=True)
dimension = passage_embeds.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(np.array(passage_embeds))
print("Faiss Index Created")

Batches:   0%|          | 0/1257 [00:00<?, ?it/s]

Faiss Index Created


In [152]:
def retrieve_top_k(query, k=5):
    query_embed = embedder.encode([query])
    D, I = faiss_index.search(np.array(query_embed), k)
    return [passages[i] for i in I[0]]

In [154]:
# Query rewriting (Gemini/Gemma via LangChain)
llm_rewriter = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest")  # Or "gemma-7b-it"
rewrite_prompt = PromptTemplate.from_template(
    """Provide several specific rewritten versions of the biomedical question, ranging from broad to precise. Output as 'Option 1', 'Option 2', etc.
Question: {question}
Rewritten:"""
)
rewrite_chain = rewrite_prompt | llm_rewriter

In [156]:
def rewrite_query(question, preferred_option="Option 2"):
    output = rewrite_chain.invoke({"question": question}).content
    for line in output.split("\n"):
        if line.strip().startswith(preferred_option):
            return line.split(":", 1)[1].strip()
    for line in output.split("\n"):
        if "Option" in line and ":" in line:
            return line.split(":", 1)[1].strip()
    return output.strip()

In [176]:
def answer_with_gemini(question, context):
    guarded_prompt = (
        "You are a biomedical expert. Answer the following question using ONLY the context below. "
        "If the answer is not present in the context, reply: \"I'm sorry, I cannot answer that question based on the provided information.\"\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return llm.invoke(guarded_prompt).content.strip()

In [287]:
import requests

GROQ_API_KEY = os.environ['GROQ_API_KEY']
  # Get at https://console.groq.com/

import requests

def answer_with_groq(question, context):
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }
    prompt = (
        "You are a biomedical expert. Answer the following question using ONLY the context below. "
        "If the answer is not present in the context, reply: \"I'm sorry, I cannot answer that question based on the provided information.\"\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    data = {
        "model": "llama3-70b-8192",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 256
    }
    resp = requests.post(url, headers=headers, json=data)
    try:
        out = resp.json()
        if 'choices' in out:
            return out['choices'][0]['message']['content'].strip(), None
        elif 'error' in out:
            # Try to extract suggested wait time
            wait_seconds = 10  # Default if not found
            if 'message' in out['error']:
                import re
                match = re.search(r'try again in ([\d\.]+)s', out['error']['message'])
                if match:
                    wait_seconds = float(match.group(1))
            #print("Groq API error:", out['error'])
            return "API error", wait_seconds
        else:
            #print("Unexpected Groq API response:", out)
            return "API call failed", 10
    except Exception as e:
        #print("Groq API response error:", e)
        #print(resp.text)
        return "API call failed", 10



In [214]:
# Complete RAG pipeline
def rag_pipeline(question, k=2, use_groq=False):
    rewritten = rewrite_query(question)
    retrieved = retrieve_top_k(rewritten, k)
    context = "\n".join(retrieved)
    if use_groq:
        answer = answer_with_groq(question, context)
    else:
        answer = answer_with_gemini(question, context)
    return {
        "question": question,
        "rewritten": rewritten,
        "context": context,
        "answer": answer
    }

## Example using Gemini

In [225]:
result = rag_pipeline("What are the symptoms of COVID-19?",use_groq=False)
print("Original Question:", result["question"])
print("\n\n")
print("Rewritten:", result["rewritten"])
print("\n\n")
print("Retrieved Context:\n", result["context"])
print("\n\n")
print("Answer:\n", result["answer"])

Original Question: What are the symptoms of COVID-19?



Rewritten: ** What is the clinical presentation of SARS-CoV-2 infection across different patient populations?



Retrieved Context:
 SARS-CoV-2, a member of the family coronaviridae, has triggered a lethal 
pandemic termed coronavirus disease 2019 (COVID-19). Pediatric patients, mainly 
from families with a cluster of infection or a history of exposure to epidemic 
areas, get infected via direct contacts or air-borne droplets. Children (aged 
below 18 years) are susceptible to COVID-19, with an average incubation period 
of about 6.5 days. Most cases present asymptomatic or common cold symptoms such 
as fever, cough, and myalgia or fatigue, which is milder than adult patients. 
Besides, most abnormal laboratory and radiologic findings in children with 
COVID-19 are non-specific. Since no specific chemotherapeutic agents have been 
approved for children, timely preventive methods could effectively forestall the 
transmission of SA

## Example using Groq

In [232]:
result = rag_pipeline("What are the symptoms of COVID-19?",use_groq=True)
print("Original Question:", result["question"])
print("\n\n")
print("Rewritten:", result["rewritten"])
print("\n\n")
print("Retrieved Context:\n", result["context"])
print("\n\n")
print("Answer:\n", result["answer"])

Original Question: What are the symptoms of COVID-19?



Rewritten: ** What is the clinical presentation of SARS-CoV-2 infection across various populations and disease severities?



Retrieved Context:
 SARS-CoV-2, a member of the family coronaviridae, has triggered a lethal 
pandemic termed coronavirus disease 2019 (COVID-19). Pediatric patients, mainly 
from families with a cluster of infection or a history of exposure to epidemic 
areas, get infected via direct contacts or air-borne droplets. Children (aged 
below 18 years) are susceptible to COVID-19, with an average incubation period 
of about 6.5 days. Most cases present asymptomatic or common cold symptoms such 
as fever, cough, and myalgia or fatigue, which is milder than adult patients. 
Besides, most abnormal laboratory and radiologic findings in children with 
COVID-19 are non-specific. Since no specific chemotherapeutic agents have been 
approved for children, timely preventive methods could effectively forestall the 
trans

# Evaluation 

In [235]:
# !pip install rouge-score evaluate
# !pip install bert-score

In [283]:
def rewrite_query_groq(question, return_wait=False):
    import os, requests, re
    url = "https://api.groq.com/openai/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {os.environ['GROQ_API_KEY']}",
        "Content-Type": "application/json"
    }
    prompt = (
        "Rewrite the following biomedical question for optimal passage retrieval. "
        "If the question is vague or not about biomedicine, reply: OUT OF SCOPE.\n"
        f"Question: {question}\nRewritten:"
    )
    data = {
        "model": "llama3-70b-8192",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 64
    }
    resp = requests.post(url, headers=headers, json=data)
    try:
        out = resp.json()
        if "choices" in out:
            rewritten = out["choices"][0]["message"]["content"].strip()
            return (rewritten, None) if return_wait else rewritten
        elif "error" in out:
            wait_seconds = 10
            if "message" in out["error"]:
                match = re.search(r'try again in ([\d\.]+)s', out["error"]["message"])
                if match:
                    wait_seconds = float(match.group(1))
            #print("Groq error:", out["error"])
            return ("API error", wait_seconds) if return_wait else "API error"
        else:
            #print("Unexpected Groq API response:", out)
            return ("API call failed", 10) if return_wait else "API call failed"
    except Exception as e:
        #print("Groq API response error:", e)
        #print(resp.text)
        return ("API call failed", 10) if return_wait else "API call failed"


In [ ]:
import os
import pandas as pd
import time
from tqdm import tqdm

CHECKPOINT_FILE = "rag_predictions_checkpoint.pkl"
SAVE_EVERY = 10  # Save every 10 questions

# Create mapping from passage text to its ID
text_to_id = dict(zip(df_passages["passage"], df_passages["id"]))

# Try to load previous progress
if os.path.exists(CHECKPOINT_FILE):
    checkpoint = pd.read_pickle(CHECKPOINT_FILE)
    processed_idxs = set(checkpoint['idx'].tolist())
    pred_answers_baseline = checkpoint['baseline'].tolist()
    pred_answers_rag = checkpoint['rag'].tolist()
    pred_passage_ids_baseline = checkpoint['baseline_pids'].tolist()
    pred_passage_ids_rag = checkpoint['rag_pids'].tolist()
    print(f"Loaded {len(processed_idxs)} previously processed entries")
else:
    processed_idxs = set()
    pred_answers_baseline = []
    pred_answers_rag = []
    pred_passage_ids_baseline = []
    pred_passage_ids_rag = []

# Loop over test set, SKIPPING already-processed ones
for idx, row in tqdm(df_test.iterrows(), total=len(df_test)):
    if idx in processed_idxs:
        continue

    question = row['question']

    # --- BASELINE ---
    retrieved_passages_base = retrieve_top_k(question, k=2)
    context_baseline = "\n".join(retrieved_passages_base)

    while True:
        answer_base, wait_time = answer_with_groq(question, context_baseline)
        if answer_base in ("API error", "API call failed"):
            time.sleep(wait_time)
            continue
        break

    retrieved_ids_base = [text_to_id[p] for p in retrieved_passages_base]

    # --- RAG ---
    while True:
        rewritten, wait_time = rewrite_query_groq(question, return_wait=True)
        if rewritten in ("API error", "API call failed"):
            time.sleep(wait_time)
            continue
        break

    retrieved_passages_rag = retrieve_top_k(rewritten, k=2)
    context_rag = "\n".join(retrieved_passages_rag)

    while True:
        answer_rag, wait_time = answer_with_groq(question, context_rag)
        if answer_rag in ("API error", "API call failed"):
            time.sleep(wait_time)
            continue
        break

    retrieved_ids_rag = [text_to_id[p] for p in retrieved_passages_rag]

    # Save results in memory
    pred_answers_baseline.append(answer_base)
    pred_answers_rag.append(answer_rag)
    pred_passage_ids_baseline.append(retrieved_ids_base)
    pred_passage_ids_rag.append(retrieved_ids_rag)
    processed_idxs.add(idx)

    # Save checkpoint every N
    if idx % SAVE_EVERY == 0:
        pd.DataFrame({
            "idx": list(processed_idxs),
            "baseline": pred_answers_baseline,
            "rag": pred_answers_rag,
            "baseline_pids": pred_passage_ids_baseline,
            "rag_pids": pred_passage_ids_rag,
        }).to_pickle(CHECKPOINT_FILE)

# Final save
pd.DataFrame({
    "idx": list(processed_idxs),
    "baseline": pred_answers_baseline,
    "rag": pred_answers_rag,
    "baseline_pids": pred_passage_ids_baseline,
    "rag_pids": pred_passage_ids_rag,
}).to_pickle(CHECKPOINT_FILE)


Loaded 311 previously processed entries


  7%|██▋                                     | 314/4719 [00:32<10:46,  6.81it/s]

In [349]:
import pandas as pd

# Load saved checkpoint
checkpoint_df = pd.read_pickle("rag_predictions_checkpoint.pkl")
print(checkpoint_df.columns)


Index(['idx', 'baseline', 'rag', 'baseline_pids', 'rag_pids'], dtype='object')


In [351]:
# Ensure gold answers are aligned with prediction indices
gold_answers = df_test.loc[checkpoint_df['idx'], 'answer'].reset_index(drop=True)

# Also align predicted outputs
baseline_answers = checkpoint_df['baseline'].reset_index(drop=True)
rag_answers = checkpoint_df['rag'].reset_index(drop=True)


In [389]:
import pandas as pd

# Load checkpoint and df_test (already available in your session)
checkpoint = pd.read_pickle("rag_predictions_checkpoint.pkl")

# Fetch gold/reference answers from df_test using stored idx
checkpoint["reference"] = checkpoint["idx"].apply(lambda i: df_test.loc[i, "answer"])

checkpoint.to_csv("routput.csv", index=False)
print("Saved as rag_evaluation_output.csv")

Saved as rag_evaluation_output.csv


In [353]:
# for gold, pred in zip(gold_answers, rag_answers):
#     print(f"Gold: {gold}\nRAG: {pred}\n---")


In [355]:
import evaluate
rouge = evaluate.load("rouge", verbose=False)
bertscore = evaluate.load("bertscore", verbose=False)


# Compute scores
rouge_result = rouge.compute(predictions=rag_answers, references=gold_answers, rouge_types=["rougeL"])
bert_result = bert.compute(predictions=rag_answers, references=gold_answers, lang="en")

print("ROUGE-L:", rouge_result["rougeL"])
print("BERT-F1:", sum(bert_result["f1"]) / len(bert_result["f1"]))


ROUGE-L: 0.19718255560488385
BERT-F1: 0.8505009488013892


In [357]:
import pandas as pd
import evaluate

# Load predictions
checkpoint = pd.read_pickle("rag_predictions_checkpoint.pkl")
df_eval = df_test.loc[checkpoint["idx"]].copy()
df_eval["baseline_pred"] = checkpoint["baseline"]
df_eval["rag_pred"] = checkpoint["rag"]

# Reference answers
references = df_eval["answer"].tolist()

# Metrics
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")


# --- Baseline Evaluation ---
print("\n--- Baseline ---")
rouge_base = rouge.compute(predictions=df_eval["baseline_pred"], references=references)
bert_base = bertscore.compute(predictions=df_eval["baseline_pred"], references=references, lang="en")


print(f"ROUGE-L: {rouge_base['rougeL']:.4f}")
print(f"BERTScore F1: {sum(bert_base['f1']) / len(bert_base['f1']):.4f}")


# --- RAG Evaluation ---
print("\n--- RAG ---")
rouge_rag = rouge.compute(predictions=df_eval["rag_pred"], references=references)
bert_rag = bertscore.compute(predictions=df_eval["rag_pred"], references=references, lang="en")


print(f"ROUGE-L: {rouge_rag['rougeL']:.4f}")
print(f"BERTScore F1: {sum(bert_rag['f1']) / len(bert_rag['f1']):.4f}")




--- Baseline ---
ROUGE-L: 0.2142
BERTScore F1: 0.8549

--- RAG ---
ROUGE-L: 0.1978
BERTScore F1: 0.8505


In [337]:
checkpoint.head(10)

,idx,baseline,rag,baseline_pids,rag_pids
0,0,"Based on the provided context, the answer is:\...","According to the context, Hirschsprung disease...","[22260, 435]","[22260, 21024]"
1,1,"Based on the provided context, the signaling m...","According to the provided context, the signali...","[23647, 32793]","[23647, 40093]"
2,2,"Yes, Papilins are secreted extracellular matri...","According to the provided context, the answer ...","[7017, 1072]","[7017, 4422]"
3,3,"I'm sorry, I cannot answer that question based...","I'm sorry, I cannot answer that question based...","[27685, 22090]","[27685, 26224]"
4,4,"I'm sorry, I cannot answer that question based...","I'm sorry, I cannot answer that question based...","[187, 17761]","[4180, 13398]"
5,5,"I'm sorry, I cannot answer that question based...","I'm sorry, I cannot answer that question based...","[10925, 6810]","[6810, 10925]"
6,6,"Based on the provided information, miRNAs-21, ...","Based on the provided context, the miRNAs that...","[25409, 12686]","[12686, 23239]"
7,7,"Based on the provided context, the answer is: ...","Based on the provided information, the answer ...","[36239, 16113]","[36239, 16113]"
8,8,"Yes, Denosumab (Prolia) has been approved by t...","Yes, Denosumab (Prolia) was approved by the FD...","[19296, 17063]","[19296, 17063]"
9,9,"I'm sorry, I cannot answer that question based...",The human genes encoding for the dishevelled p...,"[23671, 13416]","[2501, 23671]"


In [339]:
# Replace passage indices with actual passage IDs
index_to_id = dict(enumerate(df_passages["id"]))

# Replace indices in baseline_pids and rag_pids
resolved_baseline_pids = [[index_to_id[i] for i in ids] for ids in pred_passage_ids_baseline]
resolved_rag_pids = [[index_to_id[i] for i in ids] for ids in pred_passage_ids_rag]

# Save checkpoint with resolved IDs
pd.DataFrame({
    "idx": list(processed_idxs),
    "baseline": pred_answers_baseline,
    "rag": pred_answers_rag,
    "baseline_pids": resolved_baseline_pids,
    "rag_pids": resolved_rag_pids,
}).to_pickle(CHECKPOINT_FILE)

print("Final checkpoint saved with actual passage IDs!")


Final checkpoint saved with actual passage IDs!


In [347]:
# Load predictions
checkpoint = pd.read_pickle("rag_predictions_checkpoint.pkl")
checkpoint.head(5)

,idx,baseline,rag,baseline_pids,rag_pids
0,0,"Based on the provided context, the answer is:\...","According to the context, Hirschsprung disease...","[23001136, 1785632]","[23001136, 22584707]"
1,1,"Based on the provided context, the signaling m...","According to the provided context, the signali...","[23382875, 27426127]","[23382875, 34667080]"
2,2,"Yes, Papilins are secreted extracellular matri...","According to the provided context, the answer ...","[15094122, 3320045]","[15094122, 11076767]"
3,3,"I'm sorry, I cannot answer that question based...","I'm sorry, I cannot answer that question based...","[24655717, 22955974]","[24655717, 24130305]"
4,4,"I'm sorry, I cannot answer that question based...","I'm sorry, I cannot answer that question based...","[1334264, 21445329]","[10837071, 19302050]"


In [358]:
def compute_map_mrr(true_passage_ids, predicted_passage_ids):
    map_scores = []
    mrr_scores = []

    for true_ids, pred_ids in zip(true_passage_ids, predicted_passage_ids):
        ap = 0.0
        correct = 0
        rr = 0.0
        for i, pid in enumerate(pred_ids):
            if pid in true_ids:
                correct += 1
                ap += correct / (i + 1)
                if rr == 0.0:
                    rr = 1.0 / (i + 1)
        if correct > 0:
            ap /= correct
        map_scores.append(ap)
        mrr_scores.append(rr)

    return round(sum(map_scores) / len(map_scores), 4), round(sum(mrr_scores) / len(mrr_scores), 4)


In [361]:
checkpoint_df.sort_values("idx", inplace=True)

# Make sure `df_test` matches the order
df_test_sorted = df_test.reset_index().iloc[checkpoint_df['idx']].reset_index(drop=True)

# Add gold relevant_passage_ids to checkpoint_df
checkpoint_df["gold_passage_ids"] = df_test_sorted["relevant_passage_ids"]


In [363]:
import ast

# Safely convert string-represented lists into actual Python lists
checkpoint_df["gold_passage_ids"] = checkpoint_df["gold_passage_ids"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)


In [365]:
# Compute for baseline
map_base, mrr_base = compute_map_mrr(checkpoint_df["gold_passage_ids"], checkpoint_df["baseline_pids"])
print(f"Baseline → MAP: {map_base}, MRR: {mrr_base}")

# Compute for RAG
map_rag, mrr_rag = compute_map_mrr(checkpoint_df["gold_passage_ids"], checkpoint_df["rag_pids"])
print(f"RAG → MAP: {map_rag}, MRR: {mrr_rag}")


Baseline → MAP: 0.6844, MRR: 0.6844
RAG → MAP: 0.6628, MRR: 0.6628


**Recommendations:**

The baseline retriever performs slightly better than the RAG version in terms of both MAP and MRR, indicating that the passages it retrieves are more relevant (closer to the correct one) on average.


The baseline answers are slightly better than RAG on both overlap-based (ROUGE) and semantic (BERTScore) metrics.
The difference is small, but statistically meaningful depending on dataset size.

Query rewriting might be hurting retrieval precision.